In [1]:
# 1) Basic ML stack (PyTorch + HF Transformers + accelerate)
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Hugging Face + utilities
!pip install -q transformers accelerate safetensors datasets evaluate sentence-transformers

# Quantization & PEFT
!pip install -q bitsandbytes==0.40.1  # if this errors, try without version pin, or use colab's CUDA variant
!pip install -q peft

# FAISS (CPU). For GPU, try faiss-gpu if available in your runtime.
!pip install -q faiss-cpu

# Data science stack
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly

# NLP & scraping
!pip install -q spacy nltk beautifulsoup4 requests
!python -m spacy download en_core_web_sm

# Finance & I/O
!pip install -q yfinance openpyxl

# Version control & LFS
!apt-get -qq update && apt-get -qq install -y git git-lfs
!git lfs install || true

# Optional/safety/logging
!pip install -q wandb python-dotenv

# Misc (safetensors already installed above)
!pip install -q safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.7 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Git LFS initialized.


In [1]:
import torch, transformers, sys, platform
print("Python:", sys.version.splitlines()[0])
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.8.0+cu126 | CUDA available: True
CUDA device name: Tesla T4
Transformers: 4.57.1


In [2]:
import os, random, numpy as np, torch, platform
from transformers import logging

# Reproducibility seeds
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Make HF transformers quieter
logging.set_verbosity_error()

# Environment dump (log in results)
import sys
print("Python:", sys.version.splitlines()[0])
print("Platform:", platform.platform())
import transformers
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Transformers: 4.57.1
Torch: 2.8.0+cu126 CUDA: True


In [3]:
!pip install -q huggingface_hub
from huggingface_hub import login
login()  # will prompt for token in Colab

In [4]:
from huggingface_hub import whoami
whoami()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'type': 'user',
 'id': '69194d9adda5b47e2c0fbe04',
 'name': 'Puns77',
 'fullname': 'Puneet Devnani',
 'email': 'puneet.devnani1@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'periodEnd': None,
 'isPro': False,
 'avatarUrl': '/avatars/cbfcd84ac57c91ffe463f3d564ec6366.svg',
 'orgs': [{'type': 'org',
   'id': '62fe0a7a1ec28897b3c13341',
   'name': 'VIT-Bhopal',
   'fullname': 'VIT Bhopal University',
   'email': None,
   'canPay': False,
   'periodEnd': None,
   'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/QyQiKUPWUZreShzvw8rTB.png',
   'roleInOrg': 'write',
   'isEnterprise': False}],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'BloombergGPT',
   'role': 'read',
   'createdAt': '2025-11-16T05:54:40.870Z'}}}

**Load FinBERT (Finance-Specific LLM)**

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load FinBERT model
model_name = "yiyanghkust/finbert-tone"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create sentiment pipeline
finbert_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

print("FinBERT loaded successfully!")


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

FinBERT loaded successfully!


**Testing the Model with a Finance Sentence**

In [6]:
test_text = "Tesla stock drops 8% after weak Q3 earnings and reduced profit outlook."

result = finbert_pipeline(test_text)
result

[{'label': 'Negative', 'score': 0.9999970197677612}]

**Adding Reproducibility Seed (IMPORTANT)**

In [7]:
import torch, numpy as np, random, os

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

**Reproducibility Test**

In [8]:
out1 = finbert_pipeline(test_text)
out2 = finbert_pipeline(test_text)

print(out1)
print(out2)
print("Same output:", out1 == out2)

[{'label': 'Negative', 'score': 0.9999970197677612}]
[{'label': 'Negative', 'score': 0.9999970197677612}]
Same output: True


**Data Integrity Module**

**Sample Financial Dataset (News Data)**

In [9]:
import pandas as pd

data = {
    "text": [
        "Apple stock rises after strong Q4 earnings report.",
        "Tesla faces investigation over autopilot crash incidents.",
        "Google shares decline slightly amid antitrust trial.",
        "Invalid data $$##!! something wrong here...",
        None,
        "Apple stock rises after strong Q4 earnings report.",  # duplicate
    ]
}

df = pd.DataFrame(data)
df


,text
0,Apple stock rises after strong Q4 earnings rep...
1,Tesla faces investigation over autopilot crash...
2,Google shares decline slightly amid antitrust ...
3,Invalid data $$##!! something wrong here...
4,None
5,Apple stock rises after strong Q4 earnings rep...


**Removing Duplicates**

In [10]:
df_no_dup = df.drop_duplicates(subset="text")
df_no_dup.reset_index(drop=True, inplace=True)
df_no_dup

,text
0,Apple stock rises after strong Q4 earnings rep...
1,Tesla faces investigation over autopilot crash...
2,Google shares decline slightly amid antitrust ...
3,Invalid data $$##!! something wrong here...
4,None


**Detect & Remove Missing or Null Text**

In [11]:
df_clean = df_no_dup.dropna(subset=["text"])
df_clean.reset_index(drop=True, inplace=True)
df_clean

,text
0,Apple stock rises after strong Q4 earnings rep...
1,Tesla faces investigation over autopilot crash...
2,Google shares decline slightly amid antitrust ...
3,Invalid data $$##!! something wrong here...


**Detect Corrupted / Noisy Entries**

In [12]:
import re

def is_noisy(text):
    if len(text) < 20:
        return True
    if re.search(r"[^a-zA-Z0-9\s.,%\-]", text):  # strange symbols
        return True
    return False

df_clean["is_noisy"] = df_clean["text"].apply(is_noisy)
df_clean

/tmp/ipython-input-316658976.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["is_noisy"] = df_clean["text"].apply(is_noisy)


,text,is_noisy
0,Apple stock rises after strong Q4 earnings rep...,False
1,Tesla faces investigation over autopilot crash...,False
2,Google shares decline slightly amid antitrust ...,False
3,Invalid data $$##!! something wrong here...,True


In [13]:
df_final = df_clean[df_clean["is_noisy"] == False].drop(columns=["is_noisy"])
df_final.reset_index(drop=True, inplace=True)
df_final

,text
0,Apple stock rises after strong Q4 earnings rep...
1,Tesla faces investigation over autopilot crash...
2,Google shares decline slightly amid antitrust ...


**Display Integrity Summary**

In [14]:
print("Original rows:", len(df))
print("After removing duplicates:", len(df_no_dup))
print("After removing nulls:", len(df_clean))
print("After noise removal:", len(df_final))

Original rows: 6
After removing duplicates: 5
After removing nulls: 4
After noise removal: 3


**Text Embedding Drift Detection (Using Sentence Transformers)**

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
# Realistic Q1 financial news headlines
q1_news = [
    "Apple reports strong Q1 results driven by iPhone 15 sales.",
    "Tesla beats delivery expectations as global EV demand grows.",
    "Microsoft posts record cloud revenue from Azure.",
    "Amazon sees surge in AWS usage from enterprise clients.",
    "JP Morgan reports profit rise amid strong trading revenue.",
    "Google announces expansion of its AI research division.",
    "Nvidia sees higher GPU demand due to AI model training.",
    "Meta reports increased advertising revenue in Q1.",
]

# Realistic Q2 financial news headlines
q2_news = [
    "Apple warns of production delays due to supply chain disruptions in China.",
    "Tesla struggles with declining EV margins as competition increases.",
    "Microsoft faces regulatory hurdles in major acquisition deal.",
    "Amazon stock drops as inflation pressures logistics costs.",
    "JP Morgan sets aside more reserves for loan losses.",
    "Google hit with new antitrust lawsuit in the EU.",
    "Nvidia stock corrects after concerns over AI bubble valuations.",
    "Meta sees slowdown in ad revenue due to market uncertainty.",
]


**Drift Detection**

In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert to embeddings
q1_emb = embedder.encode(q1_news)
q2_emb = embedder.encode(q2_news)

# Compute mean vector for each quarter
q1_mean = np.mean(q1_emb, axis=0).reshape(1, -1)
q2_mean = np.mean(q2_emb, axis=0).reshape(1, -1)

# Cosine similarity for drift measurement
similarity = cosine_similarity(q1_mean, q2_mean)[0][0]

print("Q1–Q2 Drift Similarity Score:", similarity)


Q1–Q2 Drift Similarity Score: 0.62867934


**Fixing All Random Seeds**

In [18]:
import torch, numpy as np, random, os

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Seeds fixed.")


Seeds fixed.


**Load FinBERT (again, now with reproducibility enabled)**

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "yiyanghkust/finbert-tone"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

print("FinBERT Reloaded for reproducibility testing.")


FinBERT Reloaded for reproducibility testing.


**Picking a Financial Sentence**

In [20]:
text = "Tesla stock drops after weaker-than-expected Q2 performance."

**Running the Model Twice to Check Consistency**

In [21]:
out1 = finbert(text)
out2 = finbert(text)

print("Output 1:", out1)
print("Output 2:", out2)
print("Reproducible:", out1 == out2)


Output 1: [{'label': 'Negative', 'score': 0.9999107122421265}]
Output 2: [{'label': 'Negative', 'score': 0.9999107122421265}]
Reproducible: True


**Multi-Run Reproducibility Loop (10 Runs)**

In [22]:
outputs = []
for i in range(10):
    outputs.append(finbert(text)[0]['label'])

outputs


['Negative',
 'Negative',
 'Negative',
 'Negative',
 'Negative',
 'Negative',
 'Negative',
 'Negative',
 'Negative',
 'Negative']

In [23]:
all(x == outputs[0] for x in outputs)

True

**Log Versions(for Report)**

In [24]:
import transformers, platform, sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


Python: 3.12.12
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Transformers: 4.57.1
Torch: 2.8.0+cu126
CUDA: True


**Reproducibility Summary(for Report)**

In [25]:
import json

repro_report = {
    "seed": SEED,
    "consistent_output": out1 == out2,
    "multi_run_stability": all(x == outputs[0] for x in outputs),
    "python_version": sys.version.split()[0],
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "cuda_enabled": torch.cuda.is_available()
}

repro_report

{'seed': 42,
 'consistent_output': True,
 'multi_run_stability': True,
 'python_version': '3.12.12',
 'transformers_version': '4.57.1',
 'torch_version': '2.8.0+cu126',
 'cuda_enabled': True}

# **NLP**

**Install QA Model (FLAN-T5 Large)**

In [74]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# High accuracy financial QA (FLAN-T5)
qa_model_name = "google/flan-t5-large"

qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(qa_model_name)

flan_pipeline = pipeline(
    "text2text-generation",
    model=qa_model,
    tokenizer=qa_tokenizer,
    max_length=128,
    temperature=0.0  # deterministic output = higher reproducibility
)

print("FLAN-T5 QA Model Loaded.")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 QA Model Loaded.


**Improved Financial QA Function**

In [112]:
def finance_qa(question, context):
    prompt = f"""
    Answer the question strictly based on the context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    result = flan_pipeline(prompt)

    # result can be:
    # [{'generated_text': '...'}]  ← correct
    # ('text...',)                ← incorrect → crash

    if isinstance(result, list) and isinstance(result[0], dict):
        return {"answer": result[0]["generated_text"].strip()}

    if isinstance(result, tuple):
        return {"answer": result[0].strip()}

    return {"answer": str(result)}



**Improved Summarizer (PEGASUS Finance-Tuned)**

In [113]:
sum_model = pipeline(
    "summarization",
    model="human-centered-summarization/financial-summarization-pegasus",
    tokenizer="human-centered-summarization/financial-summarization-pegasus"
)

def finance_summary(text, min_len=40, max_len=200):
    result = sum_model(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )

    # Force dictionary return (prevents tuple errors)
    if isinstance(result, list) and isinstance(result[0], dict):
        return {"summary": result[0]["summary_text"]}

    # If it accidentally returned tuple
    if isinstance(result, tuple):
        return {"summary": str(result)}

    return {"summary": str(result)}

**Sentiment (FinBERT)**

In [114]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

sent_model = "yiyanghkust/finbert-tone"

sent_pipe = pipeline(
    "sentiment-analysis",
    model=sent_model,
    tokenizer=sent_model
)

def finance_sentiment(text):
    res = sent_pipe(text)[0]      # always a dict
    return {"label": res["label"], "score": float(res["score"])}


**Full BloombergGPT-Style Pipeline**

In [116]:
def bloomberg_gpt_pipeline(text, question=None, summary_mode="Medium"):
    # Summary mode weights
    min_len, max_len = get_summary_lengths(summary_mode)

    sent = finance_sentiment(text)
    summary = finance_summary(text, min_len=min_len, max_len=max_len)

    if question and len(question.strip()) > 0:
        qa = finance_qa(question, text)
    else:
        qa = {"answer": "No question asked."}

    # All are dictionaries → safe
    return {
        "sentiment": sent["label"],
        "sentiment_confidence": round(sent["score"], 4),
        "summary": summary["summary"],
        "qa_answer": qa["answer"]
    }


In [120]:
bloomberg_gpt_pipeline(
    "Nvidia saw Q2 growth due to AI GPU demand.",
    "Why did Nvidia grow?",
    "Detailed"
)

{'sentiment': 'Positive',
 'sentiment_confidence': 1.0,
 'summary': 'Shares of the chipmaker rise on strong AI demand growth. Nvidia says it will continue to invest in R&D to meet growing demand SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATA SALVAGEDATANRTNRTNRTNRTNRTNRTNRTNRTNRTNRTNRT picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picures picu

# **Final Integrated Pipeline**

In [121]:
# ----------------------------------------
# FINAL INTEGRATED BLOOMBERG-GPT PIPELINE
# ----------------------------------------

def bloomberg_gpt_pipeline(text, question=None):
    """
    Main function combining:
    - FinBERT Sentiment
    - PEGASUS Finance Summary
    - FLAN-T5 Large Question Answering
    """
    # 1. SENTIMENT
    sent_label, sent_conf = finance_sentiment(text)

    # 2. SUMMARY
    summary = finance_summary(text)

    # 3. QUESTION ANSWERING
    if question and len(question.strip()) > 0:
        answer = finance_qa(question, text)
    else:
        answer = "No question asked."

    return {
        "sentiment": sent_label,
        "sentiment_confidence": round(sent_conf, 4),
        "summary": summary,
        "qa_answer": answer
    }


# **Advanced UI**

In [84]:
!pip install -q gradio pandas==2.2.2

In [122]:
import gradio as gr
import pandas as pd
import os, json
from datetime import datetime

OUTPUT_DIR = "/content/bloomberg_gpt_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# SUMMARY MODE HELPER FUNCTION
# -----------------------------
def get_summary_lengths(mode):
    if mode == "Short":
        return 20, 60
    elif mode == "Medium":
        return 40, 120
    elif mode == "Detailed":
        return 80, 200
    elif mode == "Full Report":
        return 120, 300
    else:
        return 40, 120  # default


# -------------------------------------------------
# FINAL INTEGRATED BLOOMBERG GPT PIPELINE FUNCTION
# -------------------------------------------------
def bloomberg_gpt_pipeline(text, question=None, summary_mode="Medium"):

    # 1. Get summary length configuration
    min_len, max_len = get_summary_lengths(summary_mode)

    # 2. SENTIMENT
    sent = finance_sentiment(text)

    # 3. SUMMARY
    summary = finance_summary(text, min_len=min_len, max_len=max_len)

    # 4. QUESTION ANSWERING
    if question and len(question.strip()) > 0:
        qa = finance_qa(question, text)
    else:
        qa = {"answer": "No question asked."}

    return {
        "sentiment": sent["label"],
        "sentiment_confidence": round(sent["score"], 4),
        "summary": summary["summary"],
        "qa_answer": qa["answer"]
    }


# ---------------------------------------
# SINGLE RUN HANDLER FOR THE GRADIO UI
# ---------------------------------------
def ui_single_run(text, question, summary_mode):
    try:
        output = bloomberg_gpt_pipeline(text, question, summary_mode)

        result_md = f"""
### BloombergGPT Results

**🔎 Sentiment:** {output['sentiment']}
**📊 Confidence:** {output['sentiment_confidence']}

---

### 📝 Summary ({summary_mode})
{output['summary']}

---

### ❓ Answer to Your Question
{output['qa_answer']}
"""
        return result_md

    except Exception as e:
        return f"Error: {str(e)}"


# -------------------------------------
# BATCH PROCESSING (CSV UPLOAD)
# -------------------------------------
def ui_batch_run(file, text_col, question_col, summary_mode):

    df = pd.read_csv(file.name)

    min_len, max_len = get_summary_lengths(summary_mode)

    results = []

    for idx, row in df.iterrows():
        text = str(row[text_col])
        question = str(row[question_col]) if question_col in df else None

        out = bloomberg_gpt_pipeline(text, question, summary_mode)

        results.append({
            "idx": idx,
            "text": text,
            "question": question,
            "sentiment": out["sentiment"],
            "sentiment_confidence": out["sentiment_confidence"],
            "summary": out["summary"],
            "qa_answer": out["qa_answer"]
        })

    results_df = pd.DataFrame(results)

    timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    csv_path = f"{OUTPUT_DIR}/batch_results_{timestamp}.csv"
    results_df.to_csv(csv_path, index=False)

    return results_df, csv_path


# -----------------------------
# ADVANCED GRADIO UI INTERFACE
# -----------------------------
with gr.Blocks(title="BloombergGPT - Advanced Financial AI") as demo:

    gr.Markdown("# 💼 **BloombergGPT — Financial AI Assistant (Advanced UI)**")
    gr.Markdown("### Sentiment • Summary Modes • Financial QA • Batch Processing")

    with gr.Tabs():

        # ====================================================
        # SINGLE MODE TAB
        # ====================================================
        with gr.Tab("🔹 Single Analysis"):

            text_in = gr.Textbox(lines=7, label="Enter Financial News / Article")
            question_in = gr.Textbox(lines=1, label="Enter a Question (optional)")

            summary_mode = gr.Dropdown(
                ["Short", "Medium", "Detailed", "Full Report"],
                value="Medium",
                label="Summary Mode"
            )

            run_button = gr.Button("Run BloombergGPT")
            single_output = gr.Markdown()

            run_button.click(
                fn=ui_single_run,
                inputs=[text_in, question_in, summary_mode],
                outputs=single_output
            )

            gr.Markdown("### Examples")
            gr.Examples(
                examples=[
                    ["Nvidia reported strong Q2 earnings driven by AI GPU demand.", "Why did Nvidia's revenue increase?"],
                    ["Apple faces supply chain disruptions affecting iPhone production.", "What problem is Apple facing?"],
                    ["JP Morgan increases loan loss reserves amid uncertain market conditions.", "Why did JP Morgan raise reserves?"]
                ],
                inputs=[text_in, question_in]
            )

        # ====================================================
        # BATCH CSV PROCESSING TAB
        # ====================================================
        with gr.Tab("🧪 Batch CSV Processing"):

            file_in = gr.File(label="Upload CSV (must contain a 'text' column)")
            text_col_in = gr.Textbox(label="Text Column", value="text")
            question_col_in = gr.Textbox(label="Question Column (optional)", value="question")

            summary_mode_batch = gr.Dropdown(
                ["Short", "Medium", "Detailed", "Full Report"],
                value="Medium",
                label="Summary Mode (Batch)"
            )

            batch_run_button = gr.Button("Process CSV")
            batch_table = gr.DataFrame()
            batch_path_out = gr.Textbox(label="Saved CSV Path")

            batch_run_button.click(
                fn=ui_batch_run,
                inputs=[file_in, text_col_in, question_col_in, summary_mode_batch],
                outputs=[batch_table, batch_path_out]
            )

    demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7bfb9e530fb8dffdb1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [124]:
import os

# Paths where old files were stored
old_files = [
    "/content/drive/MyDrive/BloombergGPT_Project/pipeline_outputs/pipeline_meta_20251116T060038Z.json",
    "/content/drive/MyDrive/BloombergGPT_Project/pipeline_outputs/pipeline_results_20251116T060038Z.csv",
    "/content/drive/MyDrive/BloombergGPT_Project/pipeline_meta_20251116T051432Z.json",
    "/content/drive/MyDrive/BloombergGPT_Project/pipeline_results_20251116T051432Z.csv"
]

for f in old_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")
    else:
        print(f"Not found (already deleted): {f}")


Deleted: /content/drive/MyDrive/BloombergGPT_Project/pipeline_outputs/pipeline_meta_20251116T060038Z.json
Deleted: /content/drive/MyDrive/BloombergGPT_Project/pipeline_outputs/pipeline_results_20251116T060038Z.csv
Deleted: /content/drive/MyDrive/BloombergGPT_Project/pipeline_meta_20251116T051432Z.json
Deleted: /content/drive/MyDrive/BloombergGPT_Project/pipeline_results_20251116T051432Z.csv


In [127]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [128]:
import os

drive_folder = "/content/drive/MyDrive/BloombergGPT_Project"
os.makedirs(drive_folder, exist_ok=True)

drive_folder

'/content/drive/MyDrive/BloombergGPT_Project'

In [129]:
import pandas as pd

df_input = pd.DataFrame({
    "text": [
        "Nvidia reported a 40% revenue increase driven by strong demand for AI GPUs across cloud providers.",
        "Apple is facing significant supply chain disruptions affecting iPhone 15 production volume in China.",
        "JP Morgan increased loan loss reserves amid rising economic uncertainty and market volatility.",
        "Tesla achieved record quarterly production but warned that shrinking EV margins may impact profitability.",
        "Google parent Alphabet exceeded analyst expectations due to strong advertising growth and cloud revenue.",
        "Amazon AWS revenue slowed as enterprise clients cut cloud spending in anticipation of recession.",
        "Meta Platforms reported strong Q2 earnings driven by improved ad monetization and user engagement.",
        "Microsoft’s Azure division experienced rapid growth fueled by generative AI workloads.",
        "Intel warned that PC chip demand remains weak as global PC shipments continue to decline.",
        "Netflix added 9 million new subscribers as its crackdown on password sharing proved effective."
    ],
    "question": [
        "Why did Nvidia's revenue increase?",
        "What problem is Apple facing?",
        "Why did JP Morgan increase reserves?",
        "Why are Tesla's margins under pressure?",
        "Why did Alphabet beat expectations?",
        "Why is AWS growth slowing?",
        "Why did Meta have strong earnings?",
        "Why is Azure growing rapidly?",
        "Why is Intel concerned?",
        "Why did Netflix gain subscribers?"
    ]
})

df_input


,text,question
0,Nvidia reported a 40% revenue increase driven ...,Why did Nvidia's revenue increase?
1,Apple is facing significant supply chain disru...,What problem is Apple facing?
2,JP Morgan increased loan loss reserves amid ri...,Why did JP Morgan increase reserves?
3,Tesla achieved record quarterly production but...,Why are Tesla's margins under pressure?
4,Google parent Alphabet exceeded analyst expect...,Why did Alphabet beat expectations?
5,Amazon AWS revenue slowed as enterprise client...,Why is AWS growth slowing?
6,Meta Platforms reported strong Q2 earnings dri...,Why did Meta have strong earnings?
7,Microsoft’s Azure division experienced rapid g...,Why is Azure growing rapidly?
8,Intel warned that PC chip demand remains weak ...,Why is Intel concerned?
9,Netflix added 9 million new subscribers as its...,Why did Netflix gain subscribers?


No charts were generated by quickchart


In [130]:
from datetime import datetime

results = []

for idx, row in df_input.iterrows():
    text = row["text"]
    question = row["question"]

    out = bloomberg_gpt_pipeline(
        text,
        question,
        summary_mode="Detailed"
    )

    results.append({
        "idx": idx,
        "text": text,
        "question": question,
        "sentiment": out["sentiment"],
        "sentiment_confidence": out["sentiment_confidence"],
        "summary": out["summary"],
        "qa_answer": out["qa_answer"]
    })

df_results = pd.DataFrame(results)
df_results

,idx,text,question,sentiment,sentiment_confidence,summary,qa_answer
0,0,Nvidia reported a 40% revenue increase driven ...,Why did Nvidia's revenue increase?,Positive,1.0000,Revenue rose 40% on strong demand for AI GPUs....,strong demand for AI GPUs across cloud providers
1,1,Apple is facing significant supply chain disru...,What problem is Apple facing?,Negative,1.0000,iPhone 15 production halted in China due to su...,significant supply chain disruptions affecting...
2,2,JP Morgan increased loan loss reserves amid ri...,Why did JP Morgan increase reserves?,Positive,0.9665,Bank says loan loss reserves rose to 30.5 mill...,[iii]
3,3,Tesla achieved record quarterly production but...,Why are Tesla's margins under pressure?,Negative,1.0000,Elon Musk says margins may shrink as battery c...,shrinking EV margins
4,4,Google parent Alphabet exceeded analyst expect...,Why did Alphabet beat expectations?,Positive,1.0000,Google says advertising revenue rose 10% in 3Q...,strong advertising growth and cloud revenue
5,5,Amazon AWS revenue slowed as enterprise client...,Why is AWS growth slowing?,Negative,1.0000,Revenue in the third quarter fell 5% from a ye...,[iii]
6,6,Meta Platforms reported strong Q2 earnings dri...,Why did Meta have strong earnings?,Positive,1.0000,"Meta Platforms reports strong Q2 earnings, dri...",improved ad monetization and user engagement
7,7,Microsoft’s Azure division experienced rapid g...,Why is Azure growing rapidly?,Positive,1.0000,Machine learning and artificial intelligence (...,generative AI workloads
8,8,Intel warned that PC chip demand remains weak ...,Why is Intel concerned?,Negative,1.0000,Intel sees weak demand for PCs in the second h...,(iii).
9,9,Netflix added 9 million new subscribers as its...,Why did Netflix gain subscribers?,Neutral,0.9883,Netflix says it will no longer share user data...,its crackdown on password sharing proved effec...


from matplotlib import pyplot as plt
df_results['idx'].plot(kind='hist', bins=20, title='idx')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
df_results['sentiment_confidence'].plot(kind='hist', bins=20, title='sentiment_confidence')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
df_results.groupby('sentiment').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
df_results.plot(kind='scatter', x='idx', y='sentiment_confidence', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
df_results['sentiment_confidence'].plot(kind='line', figsize=(8, 4), title='sentiment_confidence')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(df_results['sentiment'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(df_results, x='idx', y='sentiment', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(df_results['sentiment'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(df_results, x='sentiment_confidence', y='sentiment', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [131]:
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
csv_path = f"{drive_folder}/final_results_{timestamp}.csv"

df_results.to_csv(csv_path, index=False)
csv_path

/tmp/ipython-input-1245407070.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


'/content/drive/MyDrive/BloombergGPT_Project/final_results_20251116T065619Z.csv'

In [132]:
import json

meta = {
    "timestamp": timestamp,
    "row_count": len(df_results),
    "summary_mode_used": "Detailed",
    "environment": env_info() if 'env_info' in globals() else {},
}

json_path = f"{drive_folder}/final_meta_{timestamp}.json"

with open(json_path, "w") as f:
    json.dump(meta, f, indent=2)

json_path

/tmp/ipython-input-36874524.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


'/content/drive/MyDrive/BloombergGPT_Project/final_meta_20251116T065619Z.json'

In [133]:
!ls -lh /content/drive/MyDrive/BloombergGPT_Project

total 11K
-rw------- 1 root root 391 Nov 16 06:56 final_meta_20251116T065619Z.json
-rw------- 1 root root 11K Nov 16 06:56 final_results_20251116T065619Z.csv


In [135]:
from google.colab import files

files.download('/content/drive/MyDrive/BloombergGPT_Project/final_meta_20251116T065619Z.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [137]:
files.download('/content/drive/MyDrive/BloombergGPT_Project/final_results_20251116T065619Z.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>